In [ ]:
import soundcard as sc
import numpy as np
import io
import wave

def configurar_loopback():
    try:
        todos = sc.all_microphones(include_loopback=True)
        loopbacks = [m for m in todos if m.isloopback]
        
        if not loopbacks:
            print("Nenhum loopback encontrado.")
            return None
        
        # Pega o primeiro loopback disponível
        device = loopbacks[0]
        print(f"Dispositivo: {device.name}")
        return device

    except Exception as e:
        print(f"Erro: {e}")
        return None

device = configurar_loopback()

In [ ]:
import soundcard as sc
import numpy as np
import io
import wave
from IPython.display import Audio, display


TAXA = 48000

def gravar_audio_memoria(duracao):
    if not device: raise Exception("Audio device error.")

    with device.recorder(samplerate=TAXA, channels=2) as rec:
        audio_np = rec.record(numframes=TAXA * duracao)

    max_val = np.max(np.abs(audio_np))
    if max_val > 0:
        audio_np = audio_np / max_val * 0.95

    audio_int16 = (audio_np * 32767).astype(np.int16)

    audio_buffer = io.BytesIO()
    wf = wave.open(audio_buffer, 'wb')
    wf.setnchannels(2)
    wf.setsampwidth(2)
    wf.setframerate(TAXA)
    wf.writeframes(audio_int16.tobytes())
    wf.close()

    return audio_buffer.getvalue()

# Teste
audio_bytes = gravar_audio_memoria(5)
with open("teste.wav", "wb") as f:
    f.write(audio_bytes)

display(Audio("teste.wav"))

In [ ]:
# importando e instanciando o shazamio
from shazamio import Shazam
shazam = Shazam()

async def reconhecer_snippet(audio_bytes):
    try:
        resultado = await shazam.recognize(audio_bytes)
        if resultado and 'track' in resultado:
            track = resultado['track']
            cover = track.get('images', {}).get('coverart', '')
            return track['title'], track['subtitle'], resultado.get('matches', [{}])[0].get('offset', 0.0), cover
    except Exception: pass
    return None, None, 0.0, ""

musica, artista, offset, url_capa = await reconhecer_snippet(audio_bytes)
print(f" Música: {musica} \n Artista: {artista} \n Offset: {offset} \n URL da Capa: {url_capa}")

In [ ]:
# importando o request e re
import requests
import re

def buscar_letra_lrclib(artista, musica):

    headers = {"User-Agent": "FrontLineLyricsApp/0.0.2"}
    padrao = re.compile(r'\[(\d{2,}):(\d{2}(?:\.\d{1,3})?)\](.*)')
    musica_limpa = re.sub(r'\([^)]*\)', '', musica).strip()
    artista_limpo = artista.split('feat.')[0].split('&')[0].strip()
    buscas = {"track_name": musica_limpa, "artist_name": artista_limpo}
    print(f'Procurando os seguintes argumentos: {buscas}')

    r = requests.get("https://lrclib.net/api/get", params=buscas, headers=headers, timeout=5)
    print(f'\nArquivo respostas do site: {json.dumps(r.json(), indent=2, ensure_ascii=False)}')
    if r.status_code == 200 and r.json().get("syncedLyrics"):
        linhas = []
        for linha in r.json()["syncedLyrics"].split('\n'):
            match = padrao.match(linha)
            if match:
                tempo = (int(match.group(1)) * 60) + float(match.group(2))
                texto = match.group(3).strip()
                if texto:
                    linhas.append({"tempo": tempo, "letra": texto})
        if linhas:
            linhas.append({"tempo": linhas[-1]["tempo"] + 5.0, "letra": "End"})
            print(f'\nLetra da música: {json.dumps(linhas, indent=2, ensure_ascii=False)}')
            return linhas


letra = buscar_letra_lrclib(artista, musica)

In [ ]:
import time

def obter_estado_atual(offset, letra):

    linha_atual, linha_anterior, linha_futura = "", "", ""
    tempo_base = time.time()
    tempo_referencia_sistema = tempo_base - offset
    tempo_decorrido = (tempo_base - tempo_referencia_sistema)

    for i, item in enumerate(letra):
        if tempo_decorrido >= item['tempo']:
            linha_atual = item['letra']
            linha_anterior = letra[i-1]['letra'] if i > 0 else ""
            linha_futura = letra[i+1]['letra'] if i + 1 < len(letra) else ""
        else:
            print("aqui") 
            break
    return linha_anterior, linha_atual, linha_futura

letra_anterior, letra_atual, letra_futura = obter_estado_atual(offset, letra)

print(f'Letra anterior: {letra_anterior}\nLetra atual: {letra_atual}\nLetra futura: {letra_futura}')

In [ ]:
from deep_translator import GoogleTranslator

def gerar_traducao(letra,idioma_alvo):

    texto_completo = "\n".join([item['letra'] for item in letra])
    texto_traduzido = GoogleTranslator(source='auto', target=idioma_alvo).translate(texto_completo).split('\n')
    linhas_traduzidas = []
    for i, item in enumerate(letra):
        letra_trad = texto_traduzido[i] if i < len(texto_traduzido) else item['letra']
        linhas_traduzidas.append({"tempo": item['tempo'], "letra": letra_trad})
    
    letra_anterior, letra_atual, letra_futura = obter_estado_atual(offset, linhas_traduzidas)

    print(f'Letra anterior: {letra_anterior}\nLetra atual: {letra_atual}\nLetra futura: {letra_futura}')

gerar_traducao(letra,"pt")